In [2]:
import torch

# Check if GPU is available
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("No GPU found. Go to Runtime > Change runtime type > T4 GPU")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.64 GB


In [3]:

LABEL2ID = {
    "evidence_based_advice": 0,
    "anecdotal_experience":  1,
    "unsupported_take":      2,
    "emotional_reaction":    3,
}

ID2LABEL    = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS  = len(LABEL2ID)
LABEL_NAMES = list(LABEL2ID.keys())

print("Label map:")
for label, idx in LABEL2ID.items():
    print(f"  {idx}  {label}")
print(f"\nTotal classes: {NUM_LABELS}")


Label map:
  0  evidence_based_advice
  1  anecdotal_experience
  2  unsupported_take
  3  emotional_reaction

Total classes: 4


In [4]:
# ── CELL 1-B: Upload CSV ─────────────────────────────────────

from google.colab import files
import io
import pandas as pd

print("▶  Click 'Choose Files' and select your labeled CSV.")
print("   Expected columns: text, label")
print()

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"\n✓  Loaded '{filename}'  —  {len(df_raw):,} rows, {df_raw.shape[1]} columns")
df_raw.head()

▶  Click 'Choose Files' and select your labeled CSV.
   Expected columns: text, label



Saving takemeter_data.csv to takemeter_data.csv

✓  Loaded 'takemeter_data.csv'  —  222 rows, 2 columns


,text,label
0,"Honestly speaking, your extracurriculars are v...",unsupported_take
1,Going from a 2.85 to a 4.0 from one year to th...,unsupported_take
2,For entrepreneurship look at Penn maybe. They ...,unsupported_take
3,Some schools don't consider your freshman year...,evidence_based_advice
4,If you are willing to pay full price your odds...,evidence_based_advice


In [5]:
REQUIRED_COLS = {"text", "label"}
missing_cols  = REQUIRED_COLS - set(df_raw.columns)
assert not missing_cols, f"Missing columns: {missing_cols}"

before = len(df_raw)
df     = df_raw.dropna(subset=["text", "label"]).copy()
dropped = before - len(df)
if dropped:
    print(f"⚠  Dropped {dropped} rows with null text or label.")

unknown = set(df["label"].unique()) - set(LABEL2ID)
assert not unknown, f"Unknown labels: {unknown}\nExpected: {set(LABEL2ID)}"

df["label_id"] = df["label"].map(LABEL2ID)

print("✓  Validation passed")
print(f"   Clean rows: {len(df):,}\n")
print("Class distribution:")
counts = df["label"].value_counts().reindex(LABEL_NAMES, fill_value=0)
for label, n in counts.items():
    bar = "█" * (n // 2)
    print(f"  {n:>4}  {label:<25}  {bar}")



✓  Validation passed
   Clean rows: 222

Class distribution:
    50  evidence_based_advice      █████████████████████████
    58  anecdotal_experience       █████████████████████████████
    76  unsupported_take           ██████████████████████████████████████
    38  emotional_reaction         ███████████████████


In [6]:
# transformers and datasets ship with Colab; install if missing
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "datasets", "scikit-learn"], check=True)

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset
import os, numpy as np

MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH       = 256
RANDOM_SEED      = 42

print(f"✓  Libraries ready")
print(f"   Model checkpoint : {MODEL_CHECKPOINT}")
print(f"   Max token length : {MAX_LENGTH}")


✓  Libraries ready
   Model checkpoint : distilbert-base-uncased
   Max token length : 256


In [7]:
df_trainval, df_test = train_test_split(
    df, test_size=0.15,
    stratify=df["label_id"], random_state=RANDOM_SEED,
)
val_frac = 0.15 / 0.85
df_train, df_val = train_test_split(
    df_trainval, test_size=val_frac,
    stratify=df_trainval["label_id"], random_state=RANDOM_SEED,
)

for split_df in [df_train, df_val, df_test]:
    split_df.reset_index(drop=True, inplace=True)

total = len(df)
print("Split summary:")
print(f"  Train : {len(df_train):>4}  ({len(df_train)/total:.0%})")
print(f"  Val   : {len(df_val):>4}  ({len(df_val)/total:.0%})")
print(f"  Test  : {len(df_test):>4}  ({len(df_test)/total:.0%})")
print(f"  Total : {total}\n")

print(f"{'Label':<25}  {'Train':>6}  {'Val':>6}  {'Test':>6}")
print("-" * 50)
for label in LABEL_NAMES:
    tr = (df_train["label"] == label).sum()
    va = (df_val["label"]   == label).sum()
    te = (df_test["label"]  == label).sum()
    print(f"  {label:<25}  {tr:>6}  {va:>6}  {te:>6}")

# Warn if any class has fewer than 5 examples in val or test
for label in LABEL_NAMES:
    for split_name, split_df in [("val", df_val), ("test", df_test)]:
        n = (split_df["label"] == label).sum()
        if n < 5:
            print(f"\n⚠  '{label}' has only {n} examples in {split_name} split.")
            print("   Consider collecting more examples of this class before training.")


Split summary:
  Train :  154  (69%)
  Val   :   34  (15%)
  Test  :   34  (15%)
  Total : 222

Label                       Train     Val    Test
--------------------------------------------------
  evidence_based_advice          35       7       8
  anecdotal_experience           40       9       9
  unsupported_take               53      12      11
  emotional_reaction             26       6       6


In [8]:
SPLIT_DIR = "/content/takemeter_splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

df_train.to_csv(f"{SPLIT_DIR}/train.csv", index=False)
df_val.to_csv(f"{SPLIT_DIR}/val.csv",     index=False)
df_test.to_csv(f"{SPLIT_DIR}/test.csv",   index=False)
print(f"✓  Splits saved to {SPLIT_DIR}/")


✓  Splits saved to /content/takemeter_splits/


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def df_to_hf_dataset(split_df):
    ds = Dataset.from_dict({
        "text":     split_df["text"].tolist(),
        "label":    split_df["label_id"].tolist(),
    })
    return ds.map(tokenize, batched=True)

print("Tokenizing splits …")
ds_train = df_to_hf_dataset(df_train)
ds_val   = df_to_hf_dataset(df_val)
ds_test  = df_to_hf_dataset(df_test)

ds_train.save_to_disk(f"{SPLIT_DIR}/ds_train")
ds_val.save_to_disk(f"{SPLIT_DIR}/ds_val")
ds_test.save_to_disk(f"{SPLIT_DIR}/ds_test")

print("✓  Tokenization complete")
print(f"   Train tokens shape : {ds_train.shape}")
print(f"   Val   tokens shape : {ds_val.shape}")
print(f"   Test  tokens shape : {ds_test.shape}")

# Spot-check: print token count distribution for train set
lengths = [sum(x) for x in ds_train["attention_mask"]]
print(f"\nTrain token-length stats (max={MAX_LENGTH}):")
print(f"  min={min(lengths)}  median={int(np.median(lengths))}  "
      f"p95={int(np.percentile(lengths,95))}  max={max(lengths)}")
if np.percentile(lengths, 95) < MAX_LENGTH * 0.6:
    print(f"  ℹ  p95 well below {MAX_LENGTH} — consider lowering MAX_LENGTH to speed up training.")

print("\nSection 2 complete. Proceed to Section 5.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/154 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/34 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/34 [00:00<?, ? examples/s]

✓  Tokenization complete
   Train tokens shape : (154, 5)
   Val   tokens shape : (34, 5)
   Test  tokens shape : (34, 5)

Train token-length stats (max=256):
  min=14  median=43  p95=100  max=150
  ℹ  p95 well below 256 — consider lowering MAX_LENGTH to speed up training.

Section 2 complete. Proceed to Section 5.


In [10]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
print("✓  groq SDK installed")


✓  groq SDK installed


In [ ]:
from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL_NAME = "llama-3.3-70b-versatile"

In [24]:
SYSTEM_PROMPT = """You are a classifier for comments from r/ApplyingToCollege.
Classify each comment into exactly one of these four labels:

evidence_based_advice
  A comment that recommends a specific action AND backs it with something
  that would still hold up as fact if the opinion framing were removed —
  a published policy, a school's own reported statistic, or a documented
  practice.

anecdotal_experience
  A comment that mainly recounts the writer's own admissions story —
  stats, decision, timeline — without aiming a recommendation at the reader.

unsupported_take
  A comment that states a confident claim, ranking, prediction, or warning
  — including advice-shaped ones — whose backing would not survive having
  the confident framing stripped away.

emotional_reaction
  A comment that is mainly the writer expressing a feeling about their own
  process, with little or no specific detail or reasoning behind it.

Decision rule for evidence_based_advice vs unsupported_take:
Strip the imperative and isolate the justification on its own.
If what remains is a specific, sourced fact (a CDS range, a published
deadline, a documented mechanism), output evidence_based_advice.
If what remains is an unfalsifiable claim about how schools treat
applicants, output unsupported_take — regardless of how directive or
numerically precise the comment sounds.

Output ONLY the label name. No explanation, no punctuation, no extra text.
Valid outputs: evidence_based_advice | anecdotal_experience | unsupported_take | emotional_reaction"""

print("System prompt set. Character count:", len(SYSTEM_PROMPT))
print("\nValid output tokens:", " | ".join(LABEL_NAMES))



System prompt set. Character count: 1530

Valid output tokens: evidence_based_advice | anecdotal_experience | unsupported_take | emotional_reaction


In [25]:
from groq import Groq
import time

client     = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL_NAME = "llama-3.3-70b-versatile"

# Reload test set from disk in case of session reset
df_test_baseline = pd.read_csv(f"{SPLIT_DIR}/test.csv")

results = []
unparseable = []

print(f"Classifying {len(df_test_baseline)} test examples with {MODEL_NAME} …\n")

for i, row in df_test_baseline.iterrows():
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": row["text"]},
            ],
            temperature=0,
            max_tokens=20,
        )
        raw = response.choices[0].message.content.strip().lower()
    except Exception as e:
        raw = ""
        print(f"  Row {i}: API error — {e}")

    # Parse: accept the label if it appears anywhere in the (short) response
    predicted = None
    for label in LABEL_NAMES:
        if label in raw:
            predicted = label
            break

    if predicted is None:
        unparseable.append({"index": i, "text": row["text"][:80], "raw_response": raw})
        predicted = "__unparseable__"

    results.append({
        "text":      row["text"],
        "true":      row["label"],
        "predicted": predicted,
        "raw":       raw,
    })

    # Progress every 10 rows
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(df_test_baseline)} done")

    time.sleep(0.1)   # light rate-limit buffer

print(f"\n✓  Classification complete")
print(f"   Unparseable responses: {len(unparseable)} / {len(df_test_baseline)} "
      f"({len(unparseable)/len(df_test_baseline):.0%})")

if len(unparseable) / len(df_test_baseline) > 0.10:
    print("\n⚠  More than 10% unparseable. Revise your prompt in Cell 5-C to")
    print("   make the expected output format clearer, then re-run this cell.")

if unparseable:
    print("\nFirst few unparseable responses:")
    for u in unparseable[:5]:
        print(f"  [{u['index']}] raw='{u['raw_response']}'  text='{u['text']}'")


Classifying 34 test examples with llama-3.3-70b-versatile …

  10/34 done
  20/34 done
  30/34 done

✓  Classification complete
   Unparseable responses: 0 / 34 (0%)


In [26]:
from sklearn.metrics import (
    accuracy_score, f1_score,
    precision_score, recall_score,
    confusion_matrix, classification_report,
)

df_results = pd.DataFrame(results)

# Exclude unparseable rows from metrics
df_scored = df_results[df_results["predicted"] != "__unparseable__"].copy()
excluded  = len(df_results) - len(df_scored)
if excluded:
    print(f"ℹ  {excluded} unparseable rows excluded from metric calculations.\n")

y_true = df_scored["true"].tolist()
y_pred = df_scored["predicted"].tolist()

acc        = accuracy_score(y_true, y_pred)
macro_f1   = f1_score(y_true, y_pred, average="macro",  labels=LABEL_NAMES, zero_division=0)
eba_prec   = precision_score(y_true, y_pred, labels=["evidence_based_advice"],
                              average="micro", zero_division=0)

print("=" * 60)
print("BASELINE METRICS")
print("=" * 60)
print(f"  Accuracy          : {acc:.3f}")
print(f"  Macro-F1          : {macro_f1:.3f}   (target ≥ 0.75)")
print(f"  EBA precision     : {eba_prec:.3f}   (target ≥ 0.80)")
print()

BASELINE METRICS
  Accuracy          : 0.794
  Macro-F1          : 0.770   (target ≥ 0.75)
  EBA precision     : 0.778   (target ≥ 0.80)



In [27]:
# Per-class table
print(f"{'Label':<25}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'Support':>8}")
print("-" * 60)
for label in LABEL_NAMES:
    p = precision_score(y_true, y_pred, labels=[label], average="micro", zero_division=0)
    r = recall_score(y_true, y_pred,    labels=[label], average="micro", zero_division=0)
    f = f1_score(y_true, y_pred,        labels=[label], average="micro", zero_division=0)
    support = sum(1 for t in y_true if t == label)
    flag = "  ← below 0.65" if f < 0.65 else ""
    print(f"  {label:<25}  {p:>6.3f}  {r:>6.3f}  {f:>6.3f}  {support:>8}{flag}")

print()

Label                        Prec     Rec      F1   Support
------------------------------------------------------------
  evidence_based_advice       0.778   0.875   0.824         8
  anecdotal_experience        0.727   0.889   0.800         9
  unsupported_take            0.900   0.818   0.857        11
  emotional_reaction          0.750   0.500   0.600         6  ← below 0.65



In [28]:
#Confusion matrix
print("Confusion matrix (rows=true, cols=predicted):")
cm = confusion_matrix(y_true, y_pred, labels=LABEL_NAMES)
header = "  " + "".join(f"{l[:6]:>10}" for l in LABEL_NAMES)
print(header)
for i, label in enumerate(LABEL_NAMES):
    row_str = "".join(f"{cm[i][j]:>10}" for j in range(len(LABEL_NAMES)))
    print(f"  {label[:6]:<6}{row_str}")

print()
print(classification_report(y_true, y_pred, labels=LABEL_NAMES, zero_division=0))



Confusion matrix (rows=true, cols=predicted):
      eviden    anecdo    unsupp    emotio
  eviden         7         0         1         0
  anecdo         0         8         0         1
  unsupp         2         0         9         0
  emotio         0         3         0         3

                       precision    recall  f1-score   support

evidence_based_advice       0.78      0.88      0.82         8
 anecdotal_experience       0.73      0.89      0.80         9
     unsupported_take       0.90      0.82      0.86        11
   emotional_reaction       0.75      0.50      0.60         6

             accuracy                           0.79        34
            macro avg       0.79      0.77      0.77        34
         weighted avg       0.80      0.79      0.79        34



In [29]:
# ── CELL 5-F: Save results & flag misclassifications ─────────────────────────

RESULTS_DIR = "/content/takemeter_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

df_results.to_csv(f"{RESULTS_DIR}/baseline_predictions.csv", index=False)

df_errors = df_scored[df_scored["true"] != df_scored["predicted"]].copy()
df_errors.to_csv(f"{RESULTS_DIR}/baseline_errors.csv", index=False)

print(f"✓  Results saved to {RESULTS_DIR}/")
print(f"   baseline_predictions.csv  ({len(df_results)} rows)")
print(f"   baseline_errors.csv       ({len(df_errors)} misclassified rows)\n")

# Boundary-pair breakdown (per planning.md Section 3)
BOUNDARY_PAIRS = [
    ("evidence_based_advice", "unsupported_take",   "HIGH-COST"),
    ("anecdotal_experience",  "emotional_reaction",  "low-cost"),
    ("anecdotal_experience",  "evidence_based_advice","medium"),
]
print("Error breakdown by boundary pair:")
for a, b, cost in BOUNDARY_PAIRS:
    ab = ((df_errors["true"] == a) & (df_errors["predicted"] == b)).sum()
    ba = ((df_errors["true"] == b) & (df_errors["predicted"] == a)).sum()
    print(f"  [{cost}]  {a[:20]} ↔ {b[:20]} : {ab + ba} errors  ({ab} + {ba})")

print()

✓  Results saved to /content/takemeter_results/
   baseline_predictions.csv  (34 rows)
   baseline_errors.csv       (7 misclassified rows)

Error breakdown by boundary pair:
  [HIGH-COST]  evidence_based_advic ↔ unsupported_take : 3 errors  (1 + 2)
  [low-cost]  anecdotal_experience ↔ emotional_reaction : 4 errors  (1 + 3)
  [medium]  anecdotal_experience ↔ evidence_based_advic : 0 errors  (0 + 0)



In [30]:
from google.colab import files
files.download(f"{RESULTS_DIR}/baseline_predictions.csv")
files.download(f"{RESULTS_DIR}/baseline_errors.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>